In [ ]:
!pip install -U qwen-tts

In [ ]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

# 1. 현재 환경 자동 감지 (Colab vs Local)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# 2. 하드웨어 가속 및 데이터 타입 자동 설정
if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# 3. 모델 로드
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype)

def generate_audio(text, lang):
    output_path = "temp_audio.wav"
    speaker = "Sohee"
    wavs, sr = model.generate_custom_voice(
        text=text,
        language=lang,
        speaker=speaker,
    )

    sf.write(output_path, wavs[0], sr)

    return output_path

# Gradio 인터페이스 구성
demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title="Qwen3-TTS Custom Voice Studio (Colab Edition)"
)

# 5. 환경에 따른 실행 옵션 자동 분기
if IS_COLAB:
    # 코랩: 외부 브라우저에서 접속하기 위해 share=True (터널링) 필수
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    # 로컬: 8000번 포트 고정, 내부 네트워크에서만 접근 (보안 유지 및 속도 빠름)
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)

In [ ]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np
import os

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
# 수정됨: scan_cache_dir 사용
from huggingface_hub import scan_cache_dir 

# ---------------------------------------------------------
# 1. 모델 ID 및 환경 변수 최상단 정의 (순서 수정)
# ---------------------------------------------------------
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

def check_model_local(repo_id):
    try:
        # 허깅페이스 로컬 캐시 디렉토리를 스캔해서 해당 모델이 있는지 확인
        hf_cache_info = scan_cache_dir()
        for repo in hf_cache_info.repos:
            if repo.repo_id == repo_id:
                return True
        return False
    except Exception as e:
        print(f"캐시 스캔 중 오류: {e}")
        return False

# 2. 모델 존재 여부에 따른 오프라인/온라인 분기
if check_model_local(model_id):
    print(f"✅ 모델 '{model_id}'가 로컬에 존재합니다. 오프라인 모드로 전환합니다.")
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    local_files_only = True
else:
    print(f"🌐 모델 '{model_id}'가 로컬에 없습니다. 다운로드를 시작합니다 (네트워크 필요).")
    os.environ['HF_HUB_OFFLINE'] = '0'
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    local_files_only = False

# ---------------------------------------------------------
# 3. 현재 환경(Colab/Local) 및 하드웨어 자동 감지
# ---------------------------------------------------------
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# ---------------------------------------------------------
# 4. 모델 로드 (동적 변수 적용)
# ---------------------------------------------------------
# 수정됨: local_files_only 옵션을 하드코딩하지 않고 변수로 받음
model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map=device, 
    dtype=dtype, 
    local_files_only=local_files_only 
)

prompt_file_path = "./voice-data/voice_prompt.pt"

# ---------------------------------------------------------
# 5. Voice 데이터 가져오기 및 캐싱
# ---------------------------------------------------------
if os.path.exists(prompt_file_path):
    print("저장된 목소리 특징 파일을 불러옵니다...")
    # 수정됨: weights_only=False 추가 (PyTorch 2.6 보안 에러 방지)
    prompt_items = torch.load(prompt_file_path, map_location=device, weights_only=False) 
else:
    print("목소리 특징 파일이 없습니다. 새로 추출을 시작합니다...")
    # 폴더가 없으면 에러가 날 수 있으니 폴더 생성 로직 추가
    os.makedirs(os.path.dirname(prompt_file_path), exist_ok=True)
    
    prompt_items = model.create_voice_clone_prompt(
        ref_audio="./audio-ref/dm_audio2.wav",
        ref_text="안녕하세요! 최신 기술 트렌드를 쉽고 빠르게 전해드리는 시간입니다. \
오늘은 개발 생태계에서 화제인 새로운 AI 최적화 방식을 알아볼까 해요. \
성능 문제로 답답하셨던 분들에게 이 아키텍처가 꼭 필요한 이유인 것이죠. \
실제 도입 시 시스템 속도가 개선되는 것을 직접 체감하실 수 있습니다. \
그럼 어떤 원리인지 지금부터 함께 살펴보겠습니다!",
        x_vector_only_mode=False,
    )
    torch.save(prompt_items, prompt_file_path)
    print(f"목소리 특징이 {prompt_file_path}에 저장되었습니다!")

# ---------------------------------------------------------
# 6. 추론 함수 및 Gradio 인터페이스 구성
# ---------------------------------------------------------
def generate_audio(text, lang):
    output_path = "./temp_audio.wav"

    # 끝음 짤림 방지를 위한 전처리 (마침표 등 추가)
    text = text.strip()
    if not text.endswith(('.', '!', '?')):
        text += '.'
    text += " ..."

    wavs, sr = model.generate_voice_clone(
        text=text,
        language=lang,
        voice_clone_prompt=prompt_items
    )

    wav = wavs[0]
    silence = np.zeros(int(sr * 0.8), dtype=wav.dtype) # 끝 여유 공간 0.8초
    wav = np.concatenate([wav, silence])

    sf.write(output_path, wav, sr)

    return output_path

demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title=f"Qwen3-TTS Voice Design Studio ({'Colab' if IS_COLAB else 'Local'} Edition)"
)

if IS_COLAB:
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)